# Типизированные решающие модели: собираем скорер руками

Ноутбук к теме [`lesson.md`](lesson.md). Здесь голова-скорер по парам «состояние —
вариант» пишется и обучается с нуля на numpy, после чего проверяется всё, что в
лекции утверждается словами: она отвечает по вариантам, которых не видела;
фиксированная голова так не умеет; перестановка вариантов не меняет ровно ничего;
рендеринг буквами меняет; для упорядоченных уровней нужна другая голова; а
температуру приходится подбирать отдельно на каждое число вариантов.

Энкодер здесь заменён случайной проекцией намеренно: проверяется **механика**
головы, а не качество конкретной сети.

In [1]:
import warnings

import numpy as np
from scipy.optimize import minimize, minimize_scalar
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore")

D_LAT, D_OBS, N_CONCEPTS, NOISE = 24, 48, 200, 0.45

def softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=axis, keepdims=True)

def make_world(seed=0):
    '''У каждого «понятия» — свой вектор смысла ψ(o); состояние видит его через M.'''
    r = np.random.default_rng(seed)
    psi = r.normal(size=(N_CONCEPTS, D_LAT))
    psi /= np.linalg.norm(psi, axis=1, keepdims=True)
    return psi, r.normal(size=(D_LAT, D_OBS)) / np.sqrt(D_LAT)

def sample_states(psi, M, concepts, n_per, seed=0):
    r = np.random.default_rng(seed)
    y = np.repeat(concepts, n_per)
    z = psi[y] + NOISE * r.normal(size=(len(y), D_LAT))
    x = z @ M + 0.2 * r.normal(size=(len(y), D_OBS))
    return x / np.linalg.norm(x, axis=1, keepdims=True), y

def make_questions(psi, M, pool, n_per, N, seed):
    '''Вопрос = состояние + N РАЗНЫХ вариантов, один из которых верный.'''
    r = np.random.default_rng(seed)
    x, y = sample_states(psi, M, pool, n_per, seed=seed)
    opts = np.empty((len(y), N), dtype=int)
    for i, gold in enumerate(y):
        opts[i, 0] = gold
        opts[i, 1:] = r.choice(pool[pool != gold], N - 1, replace=False)
    pm = np.argsort(r.random(opts.shape), axis=1)
    opts = np.take_along_axis(opts, pm, axis=1)
    return x, opts, np.argmax(opts == y[:, None], axis=1)

psi, M = make_world()
seen, unseen = np.arange(120), np.arange(120, N_CONCEPTS)
print(f"понятий всего {N_CONCEPTS}: {len(seen)} в обучении, {len(unseen)} не показываем")

понятий всего 200: 120 в обучении, 80 не показываем


## 1. Голова-скорер: двадцать строк

$s(x, o) = \varphi(x)^\top W \psi(o)$ — одна общая матрица на все варианты.
Обучение: показываем верный вариант и несколько отвлекающих, softmax по этому
маленькому множеству, кросс-энтропия.

In [2]:
def train_scorer(x, y, psi, n_opts=6, steps=8000, lr=0.8, seed=0):
    r = np.random.default_rng(seed)
    W = r.normal(size=(D_OBS, D_LAT)) * 0.01
    classes = np.unique(y)
    for _ in range(steps):
        idx = r.integers(0, len(y), 256)
        xb, yb = x[idx], y[idx]
        opts = np.empty((len(idx), n_opts), dtype=int)
        opts[:, 0] = yb
        for j in range(1, n_opts):
            opts[:, j] = r.choice(classes, len(idx))
        bad = opts[:, 1:] == yb[:, None]
        opts[:, 1:][bad] = r.choice(classes, bad.sum())
        pm = np.argsort(r.random(opts.shape), axis=1)
        opts = np.take_along_axis(opts, pm, axis=1)
        tgt = np.argmax(opts == yb[:, None], axis=1)

        E = psi[opts]                                  # (B, N, D_LAT)
        p = softmax(np.einsum("bd,bnd->bn", xb @ W, E))
        g = p.copy()
        g[np.arange(len(idx)), tgt] -= 1                # ∂CE/∂логит
        W -= lr * (xb.T @ np.einsum("bn,bnd->bd", g, E)) / len(idx)
    return W

def score(W, x, psi, opts):
    return np.einsum("bd,bnd->bn", x @ W, psi[opts])

x_tr, y_tr = sample_states(psi, M, seen, 60, seed=1)
W = train_scorer(x_tr, y_tr, psi)
print(f"обучено параметров: {W.size} (матрица {W.shape}) — и ни одного «на класс»")

обучено параметров: 1152 (матрица (48, 24)) — и ни одного «на класс»


## 2. Ответы по вариантам, которых модель не видела

Рядом — обычная голова на $K$ классов. Ей предъявляют те же вопросы.

In [3]:
head = LogisticRegression(max_iter=1000).fit(x_tr, y_tr)
lut = {c: i for i, c in enumerate(head.classes_)}

def head_accuracy(x, opts, tgt):
    known = np.isin(opts, head.classes_)
    raw = head.decision_function(x)
    lg = np.full(opts.shape, -1e9)
    for j in range(opts.shape[1]):
        ok = known[:, j]
        if ok.any():
            lg[ok, j] = raw[np.where(ok)[0], [lut[c] for c in opts[ok, j]]]
    pred = np.where(known.any(1), lg.argmax(1), -1)
    return float(np.mean(pred == tgt)), float(known.mean())

print(f"{'пул понятий':<16}{'N':>4}{'скорер':>10}{'фикс. голова':>15}"
      f"{'ей известно вариантов':>24}{'угадывание':>12}")
for tag, pool in (("знакомые", seen), ("НОВЫЕ", unseen)):
    for N in (2, 4, 8, 20, 50):
        x, opts, tgt = make_questions(psi, M, pool, 40, N, seed=100 + N)
        acc_s = float(np.mean(score(W, x, psi, opts).argmax(1) == tgt))
        acc_h, known = head_accuracy(x, opts, tgt)
        print(f"{tag:<16}{N:>4}{acc_s:>10.3f}{acc_h:>15.3f}{known:>23.0%}{1/N:>12.3f}")

пул понятий        N    скорер   фикс. голова   ей известно вариантов  угадывание
знакомые           2     0.916          0.907                   100%       0.500
знакомые           4     0.814          0.791                   100%       0.250
знакомые           8     0.684          0.661                   100%       0.125


знакомые          20     0.521          0.482                   100%       0.050
знакомые          50     0.369          0.337                   100%       0.020
НОВЫЕ              2     0.928          0.000                     0%       0.500
НОВЫЕ              4     0.812          0.000                     0%       0.250
НОВЫЕ              8     0.687          0.000                     0%       0.125


НОВЫЕ             20     0.521          0.000                     0%       0.050
НОВЫЕ             50     0.371          0.000                     0%       0.020


Скорер не различает «знакомые» и «новые»: он выучил не список классов, а правило
сопоставления состояния и варианта. Фиксированная голова на новых вариантах даёт
ровно ноль — у неё физически нет для них строки в выходном слое.

## 3. Перестановка вариантов

In [4]:
rng = np.random.default_rng(3)
x, opts, tgt = make_questions(psi, M, unseen, 120, 5, seed=3)
s = score(W, x, psi, opts)

pm = np.argsort(rng.random(opts.shape), axis=1)          # одна перестановка на вопрос
s_after = score(W, x, psi, np.take_along_axis(opts, pm, axis=1))
print(f"max|s(перестановка) − перестановка(s)| = "
      f"{np.abs(np.take_along_axis(s, pm, axis=1) - s_after).max():.1e}")

# теперь те же оценки, но варианты «отрендерены буквами»: позиция добавляет смещение
bias = np.array([0.9, 0.35, 0.0, -0.25, -0.5])
logits = s + bias
s_perm = np.take_along_axis(s, pm, axis=1)

def back(idx_in_perm):
    return np.take_along_axis(pm, idx_in_perm[:, None], 1).ravel()

variants = {
    "скорер по парам": (s.argmax(1), back(s_after.argmax(1))),
    "варианты буквами": (logits.argmax(1), back((s_perm + bias).argmax(1))),
}
b_hat = logits.mean(0) - logits.mean()                   # оценка смещения по позициям
variants["буквами + поправка"] = ((logits - b_hat).argmax(1),
                                  back(((s_perm + bias) - b_hat).argmax(1)))

print(f"\n{'способ':<22}{'точность':>10}{'ответ изменился при перестановке':>36}")
for name, (pick, pick_perm) in variants.items():
    print(f"{name:<22}{np.mean(pick == tgt):>10.3f}{np.mean(pick != pick_perm):>35.1%}")
print(f"\nистинное смещение позиций: {np.round(bias - bias.mean(), 2)}")
print(f"оценка по средним логитам: {np.round(b_hat, 2)}")

max|s(перестановка) − перестановка(s)| = 0.0e+00

способ                  точность    ответ изменился при перестановке
скорер по парам            0.771                               0.0%
варианты буквами           0.742                              16.0%
буквами + поправка         0.771                               0.8%

истинное смещение позиций: [ 0.8   0.25 -0.1  -0.35 -0.6 ]
оценка по средним логитам: [ 0.78  0.28 -0.1  -0.37 -0.59]


## 4. Вопрос типа «оценка 1…10»: порядок уровней терять нельзя

In [5]:
def ordinal_vs_flat(K=10, n_tr=300, D=40, scale=2.6, seed=0):
    r = np.random.default_rng(seed + n_tr)
    w = r.normal(size=D); w /= np.linalg.norm(w)
    X = r.normal(size=(n_tr + 4000, D))
    f = X @ w * scale + 0.8 * r.normal(size=len(X))
    th = np.quantile(f, np.linspace(0, 1, K + 1)[1:-1])
    y = np.digitize(f, th)
    Xtr, Xte, ytr, yte = X[:n_tr], X[n_tr:], y[:n_tr], y[n_tr:]

    flat = LogisticRegression(max_iter=3000).fit(Xtr, ytr)
    p_flat = np.zeros((len(yte), K)); p_flat[:, flat.classes_] = flat.predict_proba(Xte)

    def fit_ordinal(X_, y_):
        d = X_.shape[1]
        unpack = lambda v: (v[:d], np.cumsum(np.concatenate([v[d:d+1], np.exp(v[d+1:])])))
        def nll(v):
            ww, t = unpack(v)
            z = X_ @ ww
            P = 1 / (1 + np.exp(-(t[None, :] - z[:, None])))
            P = np.concatenate([np.zeros((len(z), 1)), P, np.ones((len(z), 1))], 1)
            pk = np.clip(np.diff(P, axis=1), 1e-9, 1)
            return -np.mean(np.log(pk[np.arange(len(y_)), y_])) + 1e-3 * ww @ ww
        v0 = np.concatenate([np.zeros(d), [-2.0], np.log(np.ones(K - 2) * 0.5)])
        return unpack(minimize(nll, v0, method="L-BFGS-B").x)

    ww, t = fit_ordinal(Xtr, ytr)
    z = Xte @ ww
    P = 1 / (1 + np.exp(-(t[None, :] - z[:, None])))
    P = np.concatenate([np.zeros((len(z), 1)), P, np.ones((len(z), 1))], 1)
    p_ord = np.clip(np.diff(P, axis=1), 1e-9, 1); p_ord /= p_ord.sum(1, keepdims=True)

    out = {}
    for tag, p_ in (("плоский softmax", p_flat), ("порядковая голова", p_ord)):
        pred = p_.argmax(1)
        out[tag] = (np.mean(pred == yte), np.mean(np.abs(pred - yte)),
                    np.mean(np.abs(pred - yte) >= 2), p_.shape[1])
    return out

print(f"{'обучение':>10}{'голова':>22}{'точность':>11}{'MAE в уровнях':>16}"
      f"{'ошибок на 2+':>15}")
for n_tr in (100, 300, 1000, 4000):
    res = ordinal_vs_flat(n_tr=n_tr)
    for tag, (acc, mae, big, _) in res.items():
        print(f"{n_tr:>10}{tag:>22}{acc:>11.3f}{mae:>16.3f}{big:>14.1%}")

  обучение                голова   точность   MAE в уровнях   ошибок на 2+


       100       плоский softmax      0.201           2.009         54.4%
       100     порядковая голова      0.344           1.006         25.1%
       300       плоский softmax      0.279           1.308         35.9%
       300     порядковая голова      0.442           0.741         15.2%


      1000       плоский softmax      0.387           0.887         21.7%
      1000     порядковая голова      0.483           0.641         11.0%


      4000       плоский softmax      0.432           0.748         14.8%
      4000     порядковая голова      0.477           0.658         11.8%


## 5. Температура зависит от числа вариантов

In [6]:
def ece_top(correct, conf, bins=12):
    edges = np.quantile(conf, np.linspace(0, 1, bins + 1)); edges[0], edges[-1] = 0, 1
    idx = np.clip(np.digitize(conf, edges[1:-1]), 0, bins - 1)
    return float(sum((idx == b).mean() * abs(correct[idx == b].mean() - conf[idx == b].mean())
                     for b in range(bins) if (idx == b).sum()))

def fit_T(logits, target):
    def nll(T):
        p = softmax(logits / T)
        return -np.mean(np.log(p[np.arange(len(target)), target] + 1e-12))
    return float(minimize_scalar(nll, bounds=(0.05, 20), method="bounded").x)

xv, ov, tv = make_questions(psi, M, unseen, 25, 4, seed=11)
T4 = fit_T(score(W, xv, psi, ov), tv)
print(f"температура, подобранная на вопросах с 4 вариантами: T = {T4:.2f}\n")
print(f"{'N':>4}{'точность':>11}{'уверенность':>14}{'ECE без T':>12}"
      f"{'ECE с чужой T':>15}{'своя T':>9}{'ECE со своей':>14}")
for N in (2, 4, 8, 20, 50):
    x, o, t = make_questions(psi, M, unseen, 25, N, seed=11 + N)
    lg = score(W, x, psi, o)
    T_own = fit_T(lg, t)
    row = []
    for T in (1.0, T4, T_own):
        p = softmax(lg / T)
        pred, conf = p.argmax(1), p.max(1)
        row.append((float(conf.mean()), ece_top((pred == t).astype(float), conf)))
    acc = float(np.mean(lg.argmax(1) == t))
    print(f"{N:>4}{acc:>11.3f}{row[0][0]:>14.3f}{row[0][1]:>12.4f}"
          f"{row[1][1]:>15.4f}{T_own:>9.2f}{row[2][1]:>14.4f}")

температура, подобранная на вопросах с 4 вариантами: T = 0.84

   N   точность   уверенность   ECE без T  ECE с чужой T   своя T  ECE со своей
   2      0.922         0.906      0.0203         0.0154     0.92        0.0158
   4      0.827         0.789      0.0370         0.0168     0.87        0.0169
   8      0.681         0.650      0.0380         0.0217     0.92        0.0185
  20      0.525         0.474      0.0518         0.0248     0.90        0.0204
  50      0.357         0.312      0.0485         0.0331     0.89        0.0264


## 6. Что осталось в руках

- Голова-скорер — это одна матрица и одна строка кода: `einsum("bd,bnd->bn", x @ W, psi[opts])`.
  Ни размерность, ни архитектура не зависят от числа вариантов.
- Поэтому модель отвечает по вариантам, которых не видела, ровно так же, как по знакомым,
  а фиксированная голова на них даёт ноль — у неё нет соответствующей строки.
- Перестановка вариантов не меняет скоры вообще (невязка ровно 0), если варианты
  скорятся по отдельности; при рендеринге буквами ответ переворачивается у каждого
  шестого вопроса, пока не вычтено позиционное смещение.
- Для упорядоченных уровней нужна порядковая голова: на сотне примеров она вдвое точнее
  по MAE и вдвое реже промахивается на два уровня и больше.
- Уверенность обязана падать с числом вариантов, и температуру приходится подбирать
  отдельно на каждый тип вопроса.